In [21]:
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from scipy.optimize import fsolve
import pandas as pd

## Define Leg Kinematics

Link lengths from the hexapod leg design

In [22]:
# Leg link lengths (from Leg.h)
HIP_TO_KNEE = 37.0      # a1
KNEE_TO_ANKLE = 63.54   # a2
ANKLE_TO_TIP = 200.0    # a3

MAX_REACH = HIP_TO_KNEE + KNEE_TO_ANKLE + ANKLE_TO_TIP

print(f"Link lengths:")
print(f"  Hip to Knee (a1): {HIP_TO_KNEE} mm")
print(f"  Knee to Ankle (a2): {KNEE_TO_ANKLE} mm")
print(f"  Ankle to Tip (a3): {ANKLE_TO_TIP} mm")
print(f"  Maximum reach: {MAX_REACH:.2f} mm")

Link lengths:
  Hip to Knee (a1): 37.0 mm
  Knee to Ankle (a2): 63.54 mm
  Ankle to Tip (a3): 200.0 mm
  Maximum reach: 300.54 mm


## Implement Inverse Kinematics

Based on the C++ implementation in Kinematics.cpp

In [23]:
# Range limits for each servo (in degrees)
HIP_LIMITS = (-50.0, 50.0)
KNEE_LIMITS = (-95.0, 95.0)
ANKLE_LIMITS = (-125.0, 5.0)

def inverse_kinematics(x, y, z, a1=HIP_TO_KNEE, a2=KNEE_TO_ANKLE, a3=ANKLE_TO_TIP):
    """
    Calculate inverse kinematics for the hexapod leg.
    Includes validation to prevent physically impossible configurations.
    
    Args:
        x, y, z: Target position coordinates
        a1, a2, a3: Link lengths
    
    Returns:
        tuple: (theta1, theta2, theta3) in degrees, or None if unreachable or out of limits
    """
    # Check if target is reachable
    target_dis = np.sqrt(x**2 + y**2 + z**2)
    max_leg_dis = a1 + a2 + a3
    
    if max_leg_dis < target_dis:
        return None  # Out of reach
    
    try:
        # Top view calculation, finding theta1
        theta1 = np.arctan2(y, x)
        r1 = np.sqrt(x**2 + y**2) - a1
        
        # Side view calculation, finding theta2 and theta3
        r2 = z
        phi2 = np.arctan2(r2, r1)
        r3 = np.sqrt(r1**2 + r2**2)
        
        # Clamp values for acos to prevent errors
        val1 = (a3**2 - a2**2 - r3**2) / (-2 * a2 * r3)
        val1 = np.clip(val1, -1, 1)
        phi1 = np.arccos(val1)
        
        theta2 = phi1 + phi2
        
        val2 = (r3**2 - a2**2 - a3**2) / (-2 * a2 * a3)
        val2 = np.clip(val2, -1, 1)
        phi3 = np.arccos(val2)
        
        theta3 = -(np.pi - phi3)
        
        # Convert to degrees
        theta1_deg = np.degrees(theta1)
        theta2_deg = np.degrees(theta2)
        theta3_deg = np.degrees(theta3)
        
        # Check if all angles are within physical limits
        if not (HIP_LIMITS[0] <= theta1_deg <= HIP_LIMITS[1]):
            return None
        if not (KNEE_LIMITS[0] <= theta2_deg <= KNEE_LIMITS[1]):
            return None
        if not (ANKLE_LIMITS[0] <= theta3_deg <= ANKLE_LIMITS[1]):
            return None
        
        return (theta1_deg, theta2_deg, theta3_deg)
    
    except:
        return None

## Test Specific Points

Validate key positions and their reachability

In [24]:
test_points = [
    {"x": 135.0, "y": 0.0, "z": -150.0, "desc": "Center front"},
    {"x": 135.0, "y": 100.0, "z": -150.0, "desc": "Right side"},
    {"x": 135.0, "y": -100.0, "z": -150.0, "desc": "Left side"},
    {"x": 100.0, "y": 0.0, "z": -150.0, "desc": "Close in"},
    {"x": 200.0, "y": 0.0, "z": -150.0, "desc": "Far out"},
    {"x": 135.0, "y": 0.0, "z": -100.0, "desc": "High position"},
    {"x": 135.0, "y": 0.0, "z": -200.0, "desc": "Low position"},
    {"x": 280.0, "y": 0.0, "z": -150.0, "desc": "Max X reach"},
    {"x": 50.0, "y": 0.0, "z": -150.0, "desc": "Min X reach"},
]

print("\n=== TESTING SPECIFIC POINTS ===")
for point in test_points:
    x, y, z = point["x"], point["y"], point["z"]
    distance = np.sqrt(x**2 + y**2 + z**2)
    result = inverse_kinematics(x, y, z)
    
    print(f"\n{point['desc']}: ({x:.1f}, {y:.1f}, {z:.1f})")
    print(f"  Distance: {distance:.2f} mm")
    
    if result:
        theta1, theta2, theta3 = result
        print(f"  Status: REACHABLE")
        print(f"  Angles - Hip: {theta1:.2f}°, Knee: {theta2:.2f}°, Ankle: {theta3:.2f}°")
    else:
        print(f"  Status: UNREACHABLE (out of reach)")


=== TESTING SPECIFIC POINTS ===

Center front: (135.0, 0.0, -150.0)
  Distance: 201.80 mm
  Status: REACHABLE
  Angles - Hip: 0.00°, Knee: 42.91°, Ankle: -118.00°

Right side: (135.0, 100.0, -150.0)
  Distance: 225.22 mm
  Status: REACHABLE
  Angles - Hip: 36.53°, Knee: 32.73°, Ankle: -99.91°

Left side: (135.0, -100.0, -150.0)
  Distance: 225.22 mm
  Status: REACHABLE
  Angles - Hip: -36.53°, Knee: 32.73°, Ankle: -99.91°

Close in: (100.0, 0.0, -150.0)
  Distance: 180.28 mm
  Status: UNREACHABLE (out of reach)

Far out: (200.0, 0.0, -150.0)
  Distance: 250.00 mm
  Status: REACHABLE
  Angles - Hip: 0.00°, Knee: 19.63°, Ankle: -78.58°

High position: (135.0, 0.0, -100.0)
  Distance: 168.00 mm
  Status: UNREACHABLE (out of reach)

Low position: (135.0, 0.0, -200.0)
  Distance: 241.30 mm
  Status: REACHABLE
  Angles - Hip: 0.00°, Knee: -2.71°, Ankle: -77.35°

Max X reach: (280.0, 0.0, -150.0)
  Distance: 317.65 mm
  Status: UNREACHABLE (out of reach)

Min X reach: (50.0, 0.0, -150.0)
  D

## Grid Sampling - 3D Workspace Analysis

In [37]:
# Define sampling parameters
X_MIN, X_MAX, X_STEP = 0, 300, 20.0
Y_MIN, Y_MAX, Y_STEP = -150.0, 150.0, 20.0
Z_MIN, Z_MAX, Z_STEP = -300.0, 0.0, 20.0

# Create grid
x_range = np.arange(X_MIN, X_MAX + X_STEP, X_STEP)
y_range = np.arange(Y_MIN, Y_MAX + Y_STEP, Y_STEP)
z_range = np.arange(Z_MIN, Z_MAX + Z_STEP, Z_STEP)

print(f"Grid dimensions:")
print(f"  X: {len(x_range)} points ({X_MIN} to {X_MAX})")
print(f"  Y: {len(y_range)} points ({Y_MIN} to {Y_MAX})")
print(f"  Z: {len(z_range)} points ({Z_MIN} to {Z_MAX})")
print(f"  Total points: {len(x_range) * len(y_range) * len(z_range)}")

Grid dimensions:
  X: 16 points (0 to 300)
  Y: 16 points (-150.0 to 150.0)
  Z: 16 points (-300.0 to 0.0)
  Total points: 4096


In [38]:
# Sample the workspace
reachable_points = []
unreachable_points = []
distances = []

print("Sampling workspace...")

for x in x_range:
    for y in y_range:
        for z in z_range:
            distance = np.sqrt(x**2 + y**2 + z**2)
            result = inverse_kinematics(x, y, z)
            
            if result:
                reachable_points.append((x, y, z))
                distances.append(distance)
            else:
                unreachable_points.append((x, y, z))

# Convert to arrays for easier manipulation
reachable_points = np.array(reachable_points) if reachable_points else np.array([])
unreachable_points = np.array(unreachable_points) if unreachable_points else np.array([])
distances = np.array(distances)

total_samples = len(reachable_points) + len(unreachable_points)

print("\n========== WORKSPACE STATISTICS ==========")
print(f"Total samples: {total_samples}")
print(f"Reachable points: {len(reachable_points)} ({len(reachable_points)/total_samples*100:.1f}%)")
print(f"Unreachable points: {len(unreachable_points)} ({len(unreachable_points)/total_samples*100:.1f}%)")
if len(distances) > 0:
    print(f"Min reach distance: {distances.min():.2f} mm")
    print(f"Max reach distance: {distances.max():.2f} mm")
    print(f"Avg reach distance: {distances.mean():.2f} mm")
print("==========================================")

Sampling workspace...

========== WORKSPACE STATISTICS ==========
Total samples: 4096
Reachable points: 1282 (31.3%)
Unreachable points: 2814 (68.7%)
Min reach distance: 181.38 mm
Max reach distance: 300.17 mm
Avg reach distance: 254.96 mm


## 3D Visualization of Workspace

In [39]:
fig = go.Figure()

# Plot reachable points
if len(reachable_points) > 0:
    fig.add_trace(go.Scatter3d(
        x=reachable_points[:, 0],
        y=reachable_points[:, 1],
        z=reachable_points[:, 2],
        mode='markers',
        name='Reachable',
        marker=dict(
            size=4,
            color='green',
            opacity=0.7,
            line=dict(width=0)
        ),
        text=[f"X: {x:.1f}<br>Y: {y:.1f}<br>Z: {z:.1f}<br>D: {np.sqrt(x**2+y**2+z**2):.2f}mm" 
              for x, y, z in reachable_points],
        hovertemplate='<b>Reachable</b><br>%{text}<extra></extra>'
    ))

# Plot unreachable points
if len(unreachable_points) > 0:
    fig.add_trace(go.Scatter3d(
        x=unreachable_points[:, 0],
        y=unreachable_points[:, 1],
        z=unreachable_points[:, 2],
        mode='markers',
        name='Unreachable',
        marker=dict(
            size=3,
            color='red',
            opacity=0.2,
            line=dict(width=0)
        ),
        text=[f"X: {x:.1f}<br>Y: {y:.1f}<br>Z: {z:.1f}<br>D: {np.sqrt(x**2+y**2+z**2):.2f}mm" 
              for x, y, z in unreachable_points],
        hovertemplate='<b>Unreachable</b><br>%{text}<extra></extra>'
    ))

fig.update_layout(
    title='Hexapod Leg Workspace (Interactive)',
    scene=dict(
        xaxis_title='X (mm)',
        yaxis_title='Y (mm)',
        zaxis_title='Z (mm)',
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=1.3)
        )
    ),
    hovermode='closest',
    height=800,
    width=1200
)

fig.show()

## XY Plane Visualization (at Z = -150mm)

In [40]:
# Create XY plane heatmap at Z = -150mm using Plotly
z_slice = -150.0

xy_grid = np.zeros((len(y_range), len(x_range)))
xy_text = [[f"X: {x:.0f}<br>Y: {y:.0f}<br>Z: {z_slice:.0f}" 
            for x in x_range] for y in y_range]

for i, y in enumerate(y_range):
    for j, x in enumerate(x_range):
        result = inverse_kinematics(x, y, z_slice)
        xy_grid[i, j] = 1 if result else 0

fig = go.Figure(data=go.Heatmap(
    z=xy_grid,
    x=x_range,
    y=y_range,
    colorscale='RdYlGn',
    text=xy_text,
    hovertemplate='%{text}<br>Reachable: %{z}<extra></extra>',
    colorbar=dict(
        title="Reachable",
        tickvals=[0, 1],
        ticktext=['No', 'Yes']
    )
))

fig.update_layout(
    title=f'Reachable Workspace at Z = {z_slice}mm',
    xaxis_title='X (mm)',
    yaxis_title='Y (mm)',
    height=800,
    width=1200
)

fig.show()

## Distance Heatmap - Euclidean Distance from Hip Joint

In [41]:
# Create distance heatmap at Z = -150mm using Plotly
distance_grid = np.zeros((len(y_range), len(x_range)))
distance_text = [[f"X: {x:.0f}<br>Y: {y:.0f}<br>Z: {z_slice:.0f}" 
                  for x in x_range] for y in y_range]

for i, y in enumerate(y_range):
    for j, x in enumerate(x_range):
        distance_grid[i, j] = np.sqrt(x**2 + y**2 + z_slice**2)

fig = go.Figure(data=go.Heatmap(
    z=distance_grid,
    x=x_range,
    y=y_range,
    colorscale='Viridis',
    text=distance_text,
    customdata=distance_grid,
    hovertemplate='%{text}<br>Distance: %{customdata:.2f} mm<extra></extra>',
    colorbar=dict(title="Distance (mm)")
))

fig.update_layout(
    title=f'Distance from Hip Joint at Z = {z_slice}mm',
    xaxis_title='X (mm)',
    yaxis_title='Y (mm)',
    height=800,
    width=1200
)

fig.show()

print(f"Distance range at Z = {z_slice}mm: {distance_grid.min():.2f} - {distance_grid.max():.2f} mm")

Distance range at Z = -150.0mm: 150.33 - 367.42 mm


## Z-Slice Analysis - How workspace changes with Z height

In [42]:
# Analyze reachability at different Z heights
z_test_heights = np.arange(Z_MIN, Z_MAX + 20, 20)
reachable_percentages = []
reachable_counts = []

for z in z_test_heights:
    count = 0
    total = 0
    for x in x_range:
        for y in y_range:
            total += 1
            result = inverse_kinematics(x, y, z)
            if result:
                count += 1
    
    percentage = (count / total) * 100 if total > 0 else 0
    reachable_percentages.append(percentage)
    reachable_counts.append(count)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=z_test_heights,
    y=reachable_percentages,
    mode='lines+markers',
    name='Coverage',
    line=dict(color='blue', width=2),
    marker=dict(size=8),
    text=[f"Z: {z:.1f}mm<br>Coverage: {pct:.1f}%<br>Points: {count}" 
          for z, pct, count in zip(z_test_heights, reachable_percentages, reachable_counts)],
    hovertemplate='%{text}<extra></extra>'
))

fig.update_layout(
    title='Reachable Workspace Coverage vs Z Height',
    xaxis_title='Z height (mm)',
    yaxis_title='Reachable area (%)',
    yaxis=dict(range=[0, 105]),
    height=800,
    width=1200,
    hovermode='closest',
    showlegend=True
)

fig.show()

print("\nReachable area coverage by Z height:")
for z, pct, count in zip(z_test_heights, reachable_percentages, reachable_counts):
    print(f"  Z = {z:7.1f}mm: {pct:6.1f}% ({count} points)")


Reachable area coverage by Z height:
  Z =  -300.0mm:    0.0% (0 points)
  Z =  -280.0mm:   10.2% (26 points)
  Z =  -260.0mm:   20.3% (52 points)
  Z =  -240.0mm:   28.1% (72 points)
  Z =  -220.0mm:   35.2% (90 points)
  Z =  -200.0mm:   43.0% (110 points)
  Z =  -180.0mm:   48.4% (124 points)
  Z =  -160.0mm:   44.5% (114 points)
  Z =  -140.0mm:   41.4% (106 points)
  Z =  -120.0mm:   39.1% (100 points)
  Z =  -100.0mm:   37.5% (96 points)
  Z =   -80.0mm:   34.4% (88 points)
  Z =   -60.0mm:   33.6% (86 points)
  Z =   -40.0mm:   31.2% (80 points)
  Z =   -20.0mm:   27.3% (70 points)
  Z =     0.0mm:   26.6% (68 points)
